# Markov Decision Processes

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/reinforcement-learning/01-markov-decision-processes

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A gridworld MDP

We define the environment, then solve it with **value iteration** — repeatedly applying the Bellman optimality update until the value function stops changing.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)
print('states:', env.nS, '| actions:', env.nA, '| goal:', env.goal)

## Value iteration

$V(s)\leftarrow\max_a\big[r(s,a)+\gamma V(s')\big]$ for the (deterministic) gridworld, iterated to convergence.

In [ ]:
def value_iteration(env, gamma=0.9, tol=1e-6):
    V = np.zeros(env.nS)
    for it in range(1000):
        V_new = V.copy()
        for s in range(env.nS):
            if s == env.goal: continue
            V_new[s] = max(env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA))
        if np.max(np.abs(V_new - V)) < tol:
            print(f'converged in {it} iterations'); V = V_new; break
        V = V_new
    return V

V = value_iteration(env)
print('V* grid:'); print(np.round(V.reshape(5,5), 1))

## Extract and visualize the optimal policy

The greedy policy w.r.t. $V^*$ — the best action in every cell.

In [ ]:
def greedy_policy(env, V, gamma=0.9):
    pi = np.zeros(env.nS, int)
    for s in range(env.nS):
        pi[s] = np.argmax([env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA)])
    return pi

arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = greedy_policy(env, V)
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(V.reshape(5,5), cmap='viridis')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax.text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax.set_title('Optimal value (color) + policy (arrows)'); ax.axis('off'); plt.show()

## Discounting shapes the values

In [ ]:
for g in [0.5, 0.9, 0.99]:
    Vg = value_iteration(env, gamma=g)
    print(f'gamma={g}: V(start)={Vg[0]:.2f}')

## Key takeaways

- An MDP is states, actions, transitions, rewards, and a discount $\gamma$.
- **Value iteration** applies the Bellman optimality update until $V$ converges.
- The optimal policy is **greedy** with respect to $V^*$.
- $\gamma$ sets the horizon: larger $\gamma$ values distant rewards more.

## ✏️ Your turn

### Exercise 1 — One Bellman backup step

The Bellman optimality equation for value iteration:

$$V(s) \\leftarrow \\max_a \\left[r(s,a) + \\gamma \\, V(s')\\right]$$

For a deterministic transition $s \\xrightarrow{a} s'$ this is a scalar operation.
Implement it and verify on hand-checkable fixtures.

In [ ]:
def bellman_backup(r, gamma, v_next):
    """Single Bellman backup: return r + gamma * v_next."""
    # TODO(you): one line
    ...

In [ ]:
assert abs(bellman_backup(-1, 0.9, 0.0) - (-1.0)) < 1e-9, \
    "single step to terminal state (V=0): backup = -1"
assert abs(bellman_backup(10, 0.9, 0.0) - 10.0) < 1e-9, \
    "goal transition with zero-value terminal: backup = reward"
assert abs(bellman_backup(-1, 0.9, 5.0) - 3.5) < 1e-9, \
    "bootstrap from non-zero V: -1 + 0.9*5 = 3.5"
assert abs(bellman_backup(0, 1.0, 3.0) - 3.0) < 1e-9, \
    "zero reward + gamma=1: backup equals next value"
# Zero reward everywhere: backup collapses to the discounted next value
assert abs(bellman_backup(0.0, 0.9, 10.0) - 9.0) < 1e-9, \
    "zero reward everywhere: backup = gamma * v_next = 0.9*10 = 9.0"
# Single-state MDP with a self-loop: the fixed point satisfies v = r + gamma*v,
# i.e. v* = r / (1 - gamma). Plugging the fixed point back in should reproduce it.
r_fp, gamma_fp = -1.0, 0.9
v_star = r_fp / (1 - gamma_fp)
assert abs(bellman_backup(r_fp, gamma_fp, v_star) - v_star) < 1e-9, \
    "single-state self-loop MDP: backup at the fixed point v*=r/(1-gamma) reproduces v*"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bellman_backup(r, gamma, v_next):
    return r + gamma * v_next
```

</details>

### Exercise 2 — Discount factor shapes the effective horizon

The effective horizon is approximately $\\frac{1}{1-\\gamma}$: with $\\gamma=0.9$ an agent looks
about 10 steps ahead; with $\\gamma=0.99$, about 100 steps. Verify this by computing the
discounted sum of a constant reward stream.

In [ ]:
def discounted_sum(r, gamma, n_steps):
    """Sum of discounted constant reward: r * sum(gamma^t for t in 0..n_steps-1).
    Returns the geometric-series value."""
    # TODO(you): implement (closed-form or loop, your choice)
    ...

In [ ]:
import math

# With gamma=0, only the immediate reward matters
assert abs(discounted_sum(1.0, 0.0, 100) - 1.0) < 1e-9, \
    "gamma=0 means only immediate reward counts"
# Closed-form: r * (1 - gamma^n) / (1 - gamma)
for gamma in (0.5, 0.9, 0.99):
    n = 1000
    expected = 1.0 * (1 - gamma**n) / (1 - gamma)
    assert abs(discounted_sum(1.0, gamma, n) - expected) < 1e-4, \
        f"discounted sum for gamma={gamma} should equal geometric series"
# Higher gamma means larger sum (more future rewards counted)
assert discounted_sum(1.0, 0.5, 50) < discounted_sum(1.0, 0.9, 50) < discounted_sum(1.0, 0.99, 50), \
    "higher gamma gives larger discounted sum"
# Zero reward everywhere: the discounted sum is zero regardless of gamma or horizon
assert abs(discounted_sum(0.0, 0.9, 50)) < 1e-9, \
    "zero reward everywhere: discounted sum is 0"
# A single-step horizon (n_steps=1) just returns the immediate reward
assert abs(discounted_sum(5.0, 0.9, 1) - 5.0) < 1e-9, \
    "n_steps=1: discounted sum is just the immediate reward"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def discounted_sum(r, gamma, n_steps):
    if gamma == 0.0:
        return r
    return r * (1 - gamma**n_steps) / (1 - gamma)
```

</details>

### Exercise 3 — Extra practice: vectorized Bellman update (DML #157)

[DML #157 — Implement the Bellman Equation for Value Iteration](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/157_implement-the-bellman-equation-for-value-iteration) generalizes `bellman_backup` above to a full state-value vector with **stochastic** transitions: each `transitions[s][a]` is a list of `(prob, next_state, reward, done)` tuples, and the update takes the max over actions of the expectation over outcomes.

$$V(s) \leftarrow \max_a \sum_{s',r} p(s',r\,|\,s,a)\,\big[r + \gamma\,(1-\text{done})\,V(s')\big]$$

In [ ]:
def bellman_update(V, transitions, gamma):
    """One step of value iteration over a full state-value vector.
    V: np.ndarray, shape (n_states,).
    transitions: list of dicts. transitions[s][a] is a list of
    (prob, next_state, reward, done) tuples.
    Returns: np.ndarray, updated V.
    """
    # TODO(you): for each state, take the max over actions of the expected
    # backed-up value: sum(prob * (reward + gamma * V[next_state] * (not done)))
    ...

In [ ]:
transitions = [
    {0: [(1.0, 0, 0.0, False)], 1: [(1.0, 1, 1.0, False)]},
    {0: [(1.0, 0, 0.0, False)], 1: [(1.0, 1, 1.0, True)]}
]
new_V = bellman_update(np.array([0.0, 0.0]), transitions, gamma=0.9)
assert np.allclose(np.round(new_V, 2), [1., 1.]), \
    "matches the DML #157 fixture exactly"

transitions2 = [
    {0: [(0.8, 0, 5, False), (0.2, 1, 10, False)], 1: [(1.0, 1, 2, False)]},
    {0: [(1.0, 0, 0, False)], 1: [(1.0, 1, 0, True)]}
]
new_V2 = bellman_update(np.array([0.0, 0.0]), transitions2, gamma=0.5)
assert np.allclose(np.round(new_V2, 2), [6., 0.]), \
    "matches a second DML #157 fixture with stochastic branching"

# Zero reward everywhere + single-state self-loop: one state, one action,
# reward is always 0 -- the backup is just the discounted current value.
zero_transitions = [{0: [(1.0, 0, 0.0, False)]}]
zero_V = bellman_update(np.array([5.0]), zero_transitions, gamma=0.9)
assert abs(zero_V[0] - 4.5) < 1e-9, \
    "zero reward everywhere, single-state self-loop: backup = gamma * V = 0.9*5 = 4.5"

# Deterministic vs. stochastic: a deterministic reward of 4 and a stochastic
# 50/50 mixture that *averages* to 4 must back up to the same value.
det_transitions = [{0: [(1.0, 1, 4.0, False)]}, {0: [(1.0, 1, 0.0, True)]}]
stoch_transitions = [{0: [(0.5, 1, 2.0, False), (0.5, 1, 6.0, False)]}, {0: [(1.0, 1, 0.0, True)]}]
det_V = bellman_update(np.array([0.0, 0.0]), det_transitions, gamma=0.9)
stoch_V = bellman_update(np.array([0.0, 0.0]), stoch_transitions, gamma=0.9)
assert abs(det_V[0] - stoch_V[0]) < 1e-9, \
    "deterministic reward 4 and a stochastic mixture averaging to 4 back up identically"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bellman_update(V, transitions, gamma):
    V = np.asarray(V, dtype=float)
    new_V = np.zeros_like(V)
    for s in range(len(V)):
        action_values = []
        for a, outcomes in transitions[s].items():
            q = 0.0
            for prob, next_state, reward, done in outcomes:
                q += prob * (reward + gamma * V[next_state] * (0.0 if done else 1.0))
            action_values.append(q)
        new_V[s] = max(action_values)
    return new_V
```

</details>

### Exercise 4 — Extra practice: policy evaluation on a 5×5 gridworld (DML #142)

[DML #142 — Gridworld Policy Evaluation](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/142_gridworld-policy-evaluation) asks for the **Bellman expectation** update (not the max/optimality one above): given a fixed stochastic policy — a probability over `{up, down, left, right}` per cell — iterate

$$V(s) \leftarrow \sum_a \pi(a|s)\,\big[r + \gamma\,V(s')\big]$$

to convergence. The four corners are terminal and pinned at 0. Moves that would leave the grid just bounce back to the same cell (a wall bump still costs -1).

In [ ]:
def gridworld_policy_evaluation(policy, gamma, threshold):
    """Evaluate a fixed stochastic policy on a 5x5 gridworld.
    policy: dict mapping (row, col) -> {'up': p, 'down': p, 'left': p, 'right': p}.
    gamma: discount factor.
    threshold: stop when the largest change in V is below this.
    Returns: 5x5 list of floats (state values); corners are terminal, pinned at 0.
    """
    # TODO(you): iterate the Bellman expectation update over all non-corner cells
    # until the max change is below `threshold`. Moves off the grid bounce back
    # to the same cell (still cost the -1 step reward).
    ...

In [ ]:
grid_size = 5
policy_uniform = {(i, j): {'up': 0.25, 'down': 0.25, 'left': 0.25, 'right': 0.25}
                   for i in range(grid_size) for j in range(grid_size)}
V_uniform = gridworld_policy_evaluation(policy_uniform, gamma=0.9, threshold=0.001)
assert abs(V_uniform[2][2] - (-7.0902)) < 1e-3, \
    "uniform random policy: center cell converges to about -7.09 (DML #142 example)"
assert V_uniform[0][0] == 0.0 and V_uniform[0][4] == 0.0 and V_uniform[4][0] == 0.0 and V_uniform[4][4] == 0.0, \
    "all four corners are terminal and pinned at 0"

policy_biased = {(i, j): {'up': 0.1, 'down': 0.4, 'left': 0.1, 'right': 0.4}
                  for i in range(grid_size) for j in range(grid_size)}
V_biased = gridworld_policy_evaluation(policy_biased, gamma=0.9, threshold=0.001)
assert V_biased[1][3] < 0, \
    "every non-terminal state has strictly negative value (every step costs -1)"

# Deterministic policy: always try to move 'right'. At (2, 4) that bounces off the
# wall back to (2, 4) itself every time -- a single-state self-loop whose value has
# the closed form v = r / (1 - gamma) = -1 / (1 - 0.9) = -10.
policy_det = {(i, j): {'up': 0.0, 'down': 0.0, 'left': 0.0, 'right': 1.0}
              for i in range(grid_size) for j in range(grid_size)}
V_det = gridworld_policy_evaluation(policy_det, gamma=0.9, threshold=1e-6)
assert abs(V_det[2][4] - (-10.0)) < 1e-3, \
    "deterministic policy that always bounces off the right wall: self-loop fixed point -1/(1-gamma) = -10"
print("✅ Exercise 4 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gridworld_policy_evaluation(policy, gamma, threshold):
    grid_size = 5
    actions = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
    corners = {(0, 0), (0, grid_size - 1), (grid_size - 1, 0), (grid_size - 1, grid_size - 1)}
    V = [[0.0] * grid_size for _ in range(grid_size)]
    while True:
        new_V = [row[:] for row in V]
        delta = 0.0
        for i in range(grid_size):
            for j in range(grid_size):
                if (i, j) in corners:
                    continue
                v = 0.0
                for action, prob in policy[(i, j)].items():
                    di, dj = actions[action]
                    ni = i + di if 0 <= i + di < grid_size else i
                    nj = j + dj if 0 <= j + dj < grid_size else j
                    v += prob * (-1.0 + gamma * V[ni][nj])
                new_V[i][j] = v
                delta = max(delta, abs(V[i][j] - new_V[i][j]))
        V = new_V
        if delta < threshold:
            break
    return V
```

</details>